
This notebook merges all processed modalities into a unified patient-level dataset.

Inputs:
- Clinical processed dataset
- Genomic processed dataset
- Follow-up processed dataset
- Histopathology feature embeddings

Outputs:
- multimodal_master_dataset.csv

Purpose:
Create the final patient-level dataset used for model development.

##Imports

In [138]:
# ============================================================
# IMPORT LIBRARIES
# ============================================================

import os
import glob
import numpy as np
import pandas as pd

from google.colab import drive

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

##Mount Google Drive

In [139]:
# ============================================================
# CONNECT GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Define Project Paths

In [140]:
# ============================================================
# PROJECT PATHS
# ============================================================

BASE_DIR = "/content/drive/MyDrive/TCGA_BRCA"

GENOMIC_PATH = os.path.join(
    BASE_DIR,
    "genomic_processed",
    "genomic_processed.csv"
)

CLINICAL_PATH = os.path.join(
    BASE_DIR,
    "clinical_processed",
    "clinical_processed.csv"
)

FOLLOWUP_PATH = os.path.join(
    BASE_DIR,
    "followup_processed",
    "followup_processed.csv"
)

FEATURE_DIR = os.path.join(
    BASE_DIR,
    "Features"
)

SAVE_DIR = os.path.join(
    BASE_DIR,
    "multimodal_dataset"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)

print("Project directory ready.")

Project directory ready.


##Load Clinical, Genomic and Follow-up

In [141]:
# ============================================================
# LOAD TABULAR DATA
# ============================================================

clinical = pd.read_csv(CLINICAL_PATH)

genomic = pd.read_csv(GENOMIC_PATH)

followup = pd.read_csv(FOLLOWUP_PATH)

print("="*70)
print("DATASETS LOADED")
print("="*70)

print("Clinical :", clinical.shape)
print("Genomic  :", genomic.shape)
print("FollowUp :", followup.shape)

DATASETS LOADED
Clinical : (1097, 21)
Genomic  : (1093, 1001)
FollowUp : (1095, 3)


##Load Histopathology Feature Metadata

In [142]:
# ============================================================
# LOAD FEATURE METADATA
# ============================================================

metadata_path = os.path.join(
    FEATURE_DIR,
    "features_metadata.csv"
)

metadata = pd.read_csv(metadata_path)

print("="*70)
print("FEATURE METADATA")
print("="*70)

print(metadata.shape)

display(metadata.head())

FEATURE METADATA
(369, 7)


,patient_id,num_patches,feature_shape,num_patches_total,num_patches_accepted,num_patches_rejected,num_errors
0,a1b58cd0-559e-467d-a193-96602cd3306f,12697.0,"(12697, 512)",NaN,NaN,NaN,NaN
1,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe,1989.0,"(1989, 512)",NaN,NaN,NaN,NaN
2,9132985f-ecd8-4a67-ba7b-63902801fba6,9047.0,"(9047, 512)",NaN,NaN,NaN,NaN
3,ce77af22-10b7-4480-9c93-015970268455,13609.0,"(13609, 512)",NaN,NaN,NaN,NaN
4,fcc84daa-5e54-4773-88a8-aa76727ffb6c,10695.0,"(10695, 512)",NaN,NaN,NaN,NaN


##Inspect Metadata Columns

In [143]:
# ============================================================
# METADATA INFORMATION
# ============================================================

print("="*70)
print("METADATA COLUMNS")
print("="*70)

for col in metadata.columns:
    print(col)

METADATA COLUMNS
patient_id
num_patches
feature_shape
num_patches_total
num_patches_accepted
num_patches_rejected
num_errors


##Load GDC metadata JSON

In [144]:
# ============================================================
# LOAD GDC METADATA CART JSON
# ============================================================

import json
import pandas as pd
import os


METADATA_PATH = (
    "/content/drive/MyDrive/TCGA_BRCA/"
    "metadata.cart.2026-07-20.json"
)


print("="*70)
print("LOADING GDC METADATA CART")
print("="*70)


with open(METADATA_PATH, "r") as f:
    metadata_json = json.load(f)


print("Total metadata records:")
print(len(metadata_json))


print("\nFirst record keys:")
print(metadata_json[0].keys())

LOADING GDC METADATA CART
Total metadata records:
956

First record keys:
dict_keys(['data_format', 'access', 'associated_entities', 'file_name', 'md5sum', 'file_id', 'data_type', 'submitter_id', 'data_category', 'state', 'experimental_strategy', 'file_size'])


##Convert JSON to dataframe

In [145]:
# ============================================================
# CONVERT METADATA TO DATAFRAME
# ============================================================


metadata_df = pd.json_normalize(
    metadata_json
)


print("="*70)
print("METADATA DATAFRAME")
print("="*70)


print("Shape:")
print(metadata_df.shape)


print("\nColumns:")

for col in metadata_df.columns:
    print(col)


display(
    metadata_df.head()
)

METADATA DATAFRAME
Shape:
(956, 13)

Columns:
data_format
access
associated_entities
file_name
md5sum
file_id
data_type
submitter_id
data_category
state
experimental_strategy
file_size
annotations


,data_format,access,associated_entities,file_name,md5sum,file_id,data_type,submitter_id,data_category,state,experimental_strategy,file_size,annotations
0,SVS,open,[{'entity_submitter_id': 'TCGA-E2-A14P-01Z-00-...,TCGA-E2-A14P-01Z-00-DX1.663B02FF-C64B-41A6-868...,22b18a6eea2790519592033b43f6f423,4730b23e-aea1-49a2-ba63-2231fd88b592,Slide Image,TCGA-E2-A14P-01Z-00-DX1_slide_image,Biospecimen,released,Diagnostic Slide,1597343893,NaN
1,SVS,open,[{'entity_submitter_id': 'TCGA-A7-A0CD-01Z-00-...,TCGA-A7-A0CD-01Z-00-DX1.F045B9C8-049C-41BF-843...,d148f6d71ac283e653c29ee7e77b2f23,554855d7-4e21-406b-8f9f-458b1e7c89c9,Slide Image,TCGA-A7-A0CD-01Z-00-DX1_slide_image,Biospecimen,released,Diagnostic Slide,280966835,"[{'entity_submitter_id': 'TCGA-A7-A0CD', 'note..."
2,SVS,open,[{'entity_submitter_id': 'TCGA-5L-AAT1-01Z-00-...,TCGA-5L-AAT1-01Z-00-DX1.F3449A5B-2AC4-4ED7-BF4...,12416cc41421ba836c7e11aad25594f4,4eec69ca-381b-4c17-b3e9-49492d71560e,Slide Image,TCGA-5L-AAT1-01Z-00-DX1_slide_image,Biospecimen,released,Diagnostic Slide,592769341,"[{'entity_submitter_id': 'TCGA-5L-AAT1', 'note..."
3,SVS,open,[{'entity_submitter_id': 'TCGA-A8-A09K-01Z-00-...,TCGA-A8-A09K-01Z-00-DX1.41B2DF5F-C0E1-43BB-BAA...,e984afa92734d962f2d34bdf7295c954,0dec98aa-fb71-4367-8d39-6a71ae06e442,Slide Image,TCGA-A8-A09K-01Z-00-DX1_slide_image,Biospecimen,released,Diagnostic Slide,1044914964,"[{'entity_submitter_id': 'TCGA-A8-A09K', 'note..."
4,SVS,open,[{'entity_submitter_id': 'TCGA-C8-A1HI-01Z-00-...,TCGA-C8-A1HI-01Z-00-DX1.C6D0F8B8-55ED-477F-BAF...,a40d272bed0d079189d2cce1a1db432e,1bf2c09e-854f-414f-9b5e-2ad8a5176abd,Slide Image,TCGA-C8-A1HI-01Z-00-DX1_slide_image,Biospecimen,released,Diagnostic Slide,933681451,"[{'entity_submitter_id': 'TCGA-C8-A1HI', 'note..."


##Check available patient identifiers

In [146]:
# ============================================================
# CHECK IDENTIFIER COLUMNS
# ============================================================


print("="*70)
print("IDENTIFIER COLUMNS")
print("="*70)


for col in metadata_df.columns:

    if (
        "id" in col.lower()
        or
        "case" in col.lower()
        or
        "submitter" in col.lower()
        or
        "patient" in col.lower()
    ):
        print(col)

IDENTIFIER COLUMNS
file_id
submitter_id


##Load your Features folder

In [147]:
# ============================================================
# LOAD HISTOPATHOLOGY FEATURES
# ============================================================


FEATURE_DIR = (
    "/content/drive/MyDrive/TCGA_BRCA/Features"
)


feature_files = []


for f in os.listdir(FEATURE_DIR):

    if f.endswith(".npy"):
        feature_files.append(
            f.replace(".npy","")
        )


features_uuid = pd.DataFrame(
    {
        "file_uuid": feature_files
    }
)


print("="*70)
print("FEATURE UUIDS")
print("="*70)


print(
    "Number of feature files:",
    len(features_uuid)
)


display(
    features_uuid.head()
)

FEATURE UUIDS
Number of feature files: 372


,file_uuid
0,a1b58cd0-559e-467d-a193-96602cd3306f
1,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe
2,9132985f-ecd8-4a67-ba7b-63902801fba6
3,ce77af22-10b7-4480-9c93-015970268455
4,fcc84daa-5e54-4773-88a8-aa76727ffb6c


##Compare feature UUIDs with metadata UUIDs

In [148]:
# ============================================================
# MATCH FEATURE UUIDS WITH GDC FILE UUID
# ============================================================


print("="*70)
print("MATCHING FEATURE UUIDS WITH GDC METADATA")
print("="*70)


metadata_ids = set(
    metadata_df["file_id"]
    .astype(str)
)


features_uuid["found_in_metadata"] = (
    features_uuid["file_uuid"]
    .isin(metadata_ids)
)



print(
    features_uuid["found_in_metadata"]
    .value_counts()
)


print("\nMatching percentage:")

print(
    features_uuid["found_in_metadata"]
    .mean()*100
)

MATCHING FEATURE UUIDS WITH GDC METADATA
found_in_metadata
True    372
Name: count, dtype: int64

Matching percentage:
100.0


##Extract TCGA patient IDs

In [149]:
# ============================================================
# CREATE SLIDE UUID -> PATIENT ID MAPPING
# ============================================================


print("="*70)
print("CREATING PATHOLOGY PATIENT MAPPING")
print("="*70)



mapping = metadata_df[
    [
        "file_id",
        "submitter_id"
    ]
].copy()



# Extract TCGA patient barcode

mapping["patient_id"] = (
    mapping["submitter_id"]
    .str.extract(
        r"(TCGA-[A-Z0-9]{2}-[A-Z0-9]{4})"
    )[0]
)



print("Total slides:")
print(len(mapping))


print("\nUnique patients:")
print(
    mapping["patient_id"]
    .nunique()
)


display(
    mapping.head()
)

CREATING PATHOLOGY PATIENT MAPPING
Total slides:
956

Unique patients:
901


,file_id,submitter_id,patient_id
0,4730b23e-aea1-49a2-ba63-2231fd88b592,TCGA-E2-A14P-01Z-00-DX1_slide_image,TCGA-E2-A14P
1,554855d7-4e21-406b-8f9f-458b1e7c89c9,TCGA-A7-A0CD-01Z-00-DX1_slide_image,TCGA-A7-A0CD
2,4eec69ca-381b-4c17-b3e9-49492d71560e,TCGA-5L-AAT1-01Z-00-DX1_slide_image,TCGA-5L-AAT1
3,0dec98aa-fb71-4367-8d39-6a71ae06e442,TCGA-A8-A09K-01Z-00-DX1_slide_image,TCGA-A8-A09K
4,1bf2c09e-854f-414f-9b5e-2ad8a5176abd,TCGA-C8-A1HI-01Z-00-DX1_slide_image,TCGA-C8-A1HI


##Match Histopathology Features to Patient IDs

In [150]:
# ============================================================
# MERGE FEATURES WITH PATIENT IDS
# ============================================================


feature_patient_mapping = (
    features_uuid
    .merge(
        mapping,
        left_on="file_uuid",
        right_on="file_id",
        how="left"
    )
)



print("="*70)
print("FEATURE PATIENT MAPPING")
print("="*70)


print(
    feature_patient_mapping.shape
)


print(
    "Missing patients:"
)

print(
    feature_patient_mapping["patient_id"]
    .isna()
    .sum()
)



display(
    feature_patient_mapping.head()
)

FEATURE PATIENT MAPPING
(372, 5)
Missing patients:
0


,file_uuid,found_in_metadata,file_id,submitter_id,patient_id
0,a1b58cd0-559e-467d-a193-96602cd3306f,True,a1b58cd0-559e-467d-a193-96602cd3306f,TCGA-E2-A1LB-01Z-00-DX1_slide_image,TCGA-E2-A1LB
1,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe,True,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe,TCGA-AC-A8OP-01Z-00-DX1_slide_image,TCGA-AC-A8OP
2,9132985f-ecd8-4a67-ba7b-63902801fba6,True,9132985f-ecd8-4a67-ba7b-63902801fba6,TCGA-AN-A0XP-01Z-00-DX1_slide_image,TCGA-AN-A0XP
3,ce77af22-10b7-4480-9c93-015970268455,True,ce77af22-10b7-4480-9c93-015970268455,TCGA-E9-A1R4-01Z-00-DX1_slide_image,TCGA-E9-A1R4
4,fcc84daa-5e54-4773-88a8-aa76727ffb6c,True,fcc84daa-5e54-4773-88a8-aa76727ffb6c,TCGA-AC-A2BM-01Z-00-DX1_slide_image,TCGA-AC-A2BM


##Keep only patients that actually have pathology embeddings

In [151]:
# ============================================================
# KEEP ONLY PATIENTS WITH PATHOLOGY FEATURES
# ============================================================

print("=" * 70)
print("PATIENTS WITH HISTOPATHOLOGY FEATURES")
print("=" * 70)

pathology_patients = (
    feature_patient_mapping[
        ["patient_id", "file_uuid"]
    ]
    .drop_duplicates()
    .copy()
)

print("Patients with pathology features:")
print(pathology_patients.shape[0])

display(pathology_patients.head())

PATIENTS WITH HISTOPATHOLOGY FEATURES
Patients with pathology features:
372


,patient_id,file_uuid
0,TCGA-E2-A1LB,a1b58cd0-559e-467d-a193-96602cd3306f
1,TCGA-AC-A8OP,997dbd8f-9f16-4fc2-8627-d29ee0d74ebe
2,TCGA-AN-A0XP,9132985f-ecd8-4a67-ba7b-63902801fba6
3,TCGA-E9-A1R4,ce77af22-10b7-4480-9c93-015970268455
4,TCGA-AC-A2BM,fcc84daa-5e54-4773-88a8-aa76727ffb6c


##Merge clinical

In [152]:
# ============================================================
# MERGE CLINICAL DATA
# ============================================================

multimodal = pathology_patients.merge(
    clinical,
    on="patient_id",
    how="inner"
)

print("=" * 70)
print("AFTER CLINICAL MERGE")
print("=" * 70)

print(multimodal.shape)

AFTER CLINICAL MERGE
(372, 22)


##Merge genomic

In [153]:
# ============================================================
# MERGE GENOMIC DATA
# ============================================================

multimodal = multimodal.merge(
    genomic,
    on="patient_id",
    how="inner"
)

print("=" * 70)
print("AFTER GENOMIC MERGE")
print("=" * 70)

print(multimodal.shape)

AFTER GENOMIC MERGE
(369, 1022)


##Merge follow-up

In [154]:
# ============================================================
# MERGE FOLLOW-UP DATA
# ============================================================

multimodal = multimodal.merge(
    followup,
    on="patient_id",
    how="inner"
)

print("=" * 70)
print("AFTER FOLLOW-UP MERGE")
print("=" * 70)

print(multimodal.shape)

display(multimodal.head())

AFTER FOLLOW-UP MERGE
(369, 1024)


,patient_id,file_uuid,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,histological_type_infiltratinglobularcarcinoma,histological_type_medullarycarcinoma,histological_type_metaplasticcarcinoma,histological_type_mixedhistology(pleasespecify),histological_type_mucinouscarcinoma,"histological_type_other,specify",PAM50_Her2,PAM50_LumA,PAM50_LumB,race_blackorafricanamerican,race_white,ethnicity_nothispanicorlatino,CLEC3A,CPB1,SCGB2A2,SCGB1D2,TFF1,GSTM1,PIP,S100A7,MUCL1,CYP2B7P1,ANKRD30A,PRAME,CYP4Z1,KCNJ3,AGR3,HMGCS2,SERPINA6,TFAP2B,MUC6,DHRS2,SLC30A8,UGT2B11,VSTM2A,COL2A1,C4orf7,TAT,ADIPOQ,ADH1B,CALML5,GP2,MYBPC1,GABRP,KRT14,CEACAM5,MUC5B,TFF3,C1orf64,SOX10,GRIA2,KRT5,CRABP1,GSTT1,SYT13,STAC2,CST9,LTF,KRT6B,BMPR1B,KLK11,HOXB13,CYP2A6,FABP7,NPY1R,CEACAM6,GFRA1,CRISP3,ABCC11,C20orf114,CLCA2,KIF1A,PVALB,LRP2,TCN1,AGR2,OBP2B,CP,CGA,PGR,KCNC2,SLC34A2,OLFM4,KRT6A,KLK5,MUC16,CYP4F8,PROM1,BPIL1,KRT16,DSG3,FAM5C,MSLN,SLC5A8,A2ML1,CBLN2,SERPINA11,NLRP2,PIGR,PTPRT,AQP5,KRT81,ELF5,KRT17,CA9,VGLL1,BEX1,CST1,TUSC5,SLC6A4,LBP,KLK6,KLK10,CYP4Z2P,PPP1R1B,NXPH1,SLITRK6,KLHDC7A,LRRC31,ESR1,C10orf82,PPP2R2C,EEF1A2,SYT9,DCD,ANKRD30B,SCGB2A1,ORM1,CALML3,GRPR,FABP4,CHAD,TNNT1,CST5,SORCS1,MAGEA6,KLK7,ASCL1,MUC2,CLIC6,PDZK1,CDC20B,CXCL17,SLC6A14,RPS28,PEG10,DIO1,CARTPT,FOXI1,NBPF4,SPAG6,PYDC1,C7,SDR16C5,CASP14,CNTNAP2,ABCC8,WNK4,MUC15,CIDEC,LCN2,S100A7A,PPAN-P2RY11,SCUBE2,MIA,S100A8,ABCA12,SCGB3A1,PGLYRP2,FOLR1,DSC3,C2orf54,CAPN8,IGSF1,MSMB,CXCL13,S100A9,LRP1B,MAGEA3,SERPINA5,TMPRSS4,LY6D,TRH,SERPINB5,GLYATL2,SLC7A4,HS6ST3,SLC5A1,PON3,MMP1,ERBB4,SLC44A4,SFRP1,NBPF6,PLA2G2A,COL17A1,ALB,ZIC1,S100P,ROBO2,RIMS4,ORM2,CYP4F22,WIF1,NEK10,NCRNA00052,GLRA3,TMC5,ABCC13,IL20,ROPN1,PI16,LPPR3,ELOVL2,INSM1,TSPAN8,CCL19,SLC9A2,SLC7A2,LEP,PI3,LOC728606,FREM2,MS4A15,WFDC2,KNDC1,FAM3B,HGD,ATP6V0A4,WDR72,FGB,CAPN6,CHST8,ATP13A5,NKAIN1,TRPA1,SYT1,NCCRP1,FAM5B,PTPRZ1,ANXA8,CYP4B1,HORMAD1,VTCN1,FOXJ1,CCL21,SYTL5,KLK8,ACTL8,PKP1,CR2,GRB14,SPDYC,KRT15,PLIN1,AGTR1,CHGB,BCAS1,C16orf89,GLDC,TPRG1,BBOX1,TMPRSS6,GPD1,HOTAIR,CYP4X1,TUBA3D,ART3,SYT8,LRG1,GPR98,NAT1,CYP2A7,MS4A1,FSIP1,FAM196A,CIDEA,GSTA1,TRIM29,SLPI,PROL1,NDP,DLK1,CHIT1,ZIC2,FLJ45983,PAX7,ROPN1B,SOSTDC1,HEPACAM2,PPP4R4,PCSK1N,ALOX15B,PNMT,ATRNL1,IGDCC3,CST4,TF,MARCO,PSCA,MMP12,RGS22,ANXA8L2,DSG1,C2CD4A,SLC13A2,LOC100271831,GPR26,AKR7A3,ONECUT2,GABRB3,HOXC11,PRODH,GJB1,KRT23,KRT6C,RBM24,C1orf173,C19orf33,C3orf57,GRIK3,FOXA1,TRPV6,THRSP,UGT2B15,ARHGAP36,DARC,SAA2,SHISA9,KRT4,TMPRSS3,FUT3,VSIG2,IGJ,POF1B,PPP1R14C,SAA1,VIPR2,TPSG1,PRSS21,TDRD1,HBA1,PLIN4,TFPI2,KRT13,AREG,TTYH1,SLC26A3,SLC28A3,PLCH1,MMP13,CCNO,AR,DLX2,DKK1,CXADRP3,SYNPO2L,AFF3,ATP13A4,S100A2,AKR1B10,PI15,SPAG17,NTNG1,FAM83E,SBSN,PCDHA11,ANKRD43,ZNF385B,NPY5R,F7,HSD17B2,DCX,NELL2,MAPK4,FGG,PCDHA12,IYD,SOX2,MYT1,CACNG4,DCDC2,FAM83A,EPYC,ADCY5,TSIX,CHI3L2,PHGR1,PCDH10,WT1,SHISA2,KCNG1,PP14571,EDN3,HPX,RIMS2,COL11A1,CD177,TLX1,PCK1,GLYATL1,CYP4F11,SEZ6L,LOC642587,S100A1,TTC36,MMP7,PART1,NEURL,TBC1D3G,CNTFR,GABBR2,CRYM,TOX3,PPP1R1A,SCRG1,PLA2G2D,LGR6,RET,SLC15A1,ZFP57,SMOC1,HOXC10,TUBA3E,UGT2B28,MKRN3,HRASLS5,SLC6A11,CXCL14,CHGA,PLA2G3,KCNF1,RHCG,CHRDL1,TSPAN1,MKX,BCL11A,FBN3,MS4A8B,SLC6A15,NKX2-2,MAGEA1,FUT6,MAPT,CTNND2,CPLX2,AKR1C2,DOK7,UGT8,PCSK1,LOC389033,CECR2,APOD,OGN,RASAL1,DACH1,ABCC12,HP,TMEM40,MMP10,CPA6,ABCA8,FAT2,SCNN1G,DEFB1,LY6K,C2orf40,SCARA5,FGFBP1,NOVA1,CLGN,FGF10,SCGN,PIK3C2G,CCL18,RBP4,GRM4,TPSD1,ADAMTS15,CD79A,C1orf106,TNFSF12-TNFSF13,FLT3,AQP7,PAH,GPRIN2,PLIN5,NMU,IRX4,ADAMTS19,SHC4,CLSTN2,GATA4,GJB3,B3GNT3,FCRL5,TMEM132C,SLC1A1,EMX1,PCOLCE2,DLGAP1,PSAT1,MAOB,COL9A3,AFP,ADAM6,LOC339535,IRX1,LOC440905,SEMA3E,CD300LG,SERHL2,GNG4,CAPN9,MYO3B,LOC84740,PDZK1IP1,SLC44A5,CXorf61,DNALI1,HAPLN1,PAX2,SLC19A3,CD19,DLX1,SPDEF,C6,SERPINB2,MAGEA4,IL12RB2,UNC5A,CAPN13,CHRM1,COCH,PEG3,CKMT1A,TMEM213,FERMT1,SPINK8,SOX11,NELL1,PADI3,TNNI3,CXCL9,CHI3L1,PCDHA10,PTCHD1,FUT9,CA8,LOC145837,LRRC26,GAL,CWH43,CA12,DNER,RNF183,L1CAM,SLC27A6,STC2,DPYSL5,CDSN,ACTA1,TRIM58,PCDHA6,M

##Check missing values

In [155]:
# ============================================================
# MULTIMODAL QUALITY CONTROL
# ============================================================

print("=" * 70)
print("MULTIMODAL QUALITY CONTROL")
print("=" * 70)

print("Patients:")
print(multimodal["patient_id"].nunique())

print()

print("Duplicate patients:")
print(multimodal["patient_id"].duplicated().sum())

print()

print("Missing values per column:")

missing = (
    multimodal
    .isna()
    .sum()
)

display(
    missing[
        missing > 0
    ]
    .sort_values(ascending=False)
)

MULTIMODAL QUALITY CONTROL
Patients:
357

Duplicate patients:
12

Missing values per column:


,0


##Remove duplicate patients

In [156]:
# ============================================================
# REMOVE DUPLICATE PATIENTS
# ============================================================

print("=" * 70)
print("REMOVING DUPLICATE PATIENTS")
print("=" * 70)

print("Before:")
print(multimodal.shape)

multimodal = (
    multimodal
    .sort_values("file_uuid")
    .drop_duplicates(
        subset="patient_id",
        keep="first"
    )
    .reset_index(drop=True)
)

print("\nAfter:")
print(multimodal.shape)

print("\nUnique patients:")
print(multimodal["patient_id"].nunique())

REMOVING DUPLICATE PATIENTS
Before:
(369, 1024)

After:
(357, 1024)

Unique patients:
357


##Event distribution

In [157]:
# ============================================================
# EVENT DISTRIBUTION
# ============================================================

print("=" * 70)
print("SURVIVAL LABELS")
print("=" * 70)

print(multimodal["event"].value_counts())

print()

print("Event ratio:")

print(
    multimodal["event"].mean()
)

SURVIVAL LABELS
event
0    266
1     91
Name: count, dtype: int64

Event ratio:
0.2549019607843137


##Save the master dataset

In [158]:
# ============================================================
# SAVE MASTER DATASET
# ============================================================

master_path = os.path.join(
    SAVE_DIR,
    "multimodal_master_dataset.csv"
)

multimodal.to_csv(
    master_path,
    index=False
)

print("=" * 70)
print("MASTER DATASET SAVED")
print("=" * 70)

print(master_path)

MASTER DATASET SAVED
/content/drive/MyDrive/TCGA_BRCA/multimodal_dataset/multimodal_master_dataset.csv


##TESTING

##Reload the saved dataset

In [159]:
# ============================================================
# LOAD FINAL MASTER DATASET
# ============================================================

master_path = os.path.join(
    SAVE_DIR,
    "multimodal_master_dataset.csv"
)

master = pd.read_csv(master_path)

print("=" * 70)
print("FINAL MASTER DATASET")
print("=" * 70)

print(master.shape)

display(master.head())

FINAL MASTER DATASET
(357, 1024)


,patient_id,file_uuid,years_to_birth,Tumor_purity,pathologic_stage,pathology_T_stage,pathology_N_stage,pathology_M_stage,number_of_lymph_nodes,radiation_therapy,histological_type_infiltratinglobularcarcinoma,histological_type_medullarycarcinoma,histological_type_metaplasticcarcinoma,histological_type_mixedhistology(pleasespecify),histological_type_mucinouscarcinoma,"histological_type_other,specify",PAM50_Her2,PAM50_LumA,PAM50_LumB,race_blackorafricanamerican,race_white,ethnicity_nothispanicorlatino,CLEC3A,CPB1,SCGB2A2,SCGB1D2,TFF1,GSTM1,PIP,S100A7,MUCL1,CYP2B7P1,ANKRD30A,PRAME,CYP4Z1,KCNJ3,AGR3,HMGCS2,SERPINA6,TFAP2B,MUC6,DHRS2,SLC30A8,UGT2B11,VSTM2A,COL2A1,C4orf7,TAT,ADIPOQ,ADH1B,CALML5,GP2,MYBPC1,GABRP,KRT14,CEACAM5,MUC5B,TFF3,C1orf64,SOX10,GRIA2,KRT5,CRABP1,GSTT1,SYT13,STAC2,CST9,LTF,KRT6B,BMPR1B,KLK11,HOXB13,CYP2A6,FABP7,NPY1R,CEACAM6,GFRA1,CRISP3,ABCC11,C20orf114,CLCA2,KIF1A,PVALB,LRP2,TCN1,AGR2,OBP2B,CP,CGA,PGR,KCNC2,SLC34A2,OLFM4,KRT6A,KLK5,MUC16,CYP4F8,PROM1,BPIL1,KRT16,DSG3,FAM5C,MSLN,SLC5A8,A2ML1,CBLN2,SERPINA11,NLRP2,PIGR,PTPRT,AQP5,KRT81,ELF5,KRT17,CA9,VGLL1,BEX1,CST1,TUSC5,SLC6A4,LBP,KLK6,KLK10,CYP4Z2P,PPP1R1B,NXPH1,SLITRK6,KLHDC7A,LRRC31,ESR1,C10orf82,PPP2R2C,EEF1A2,SYT9,DCD,ANKRD30B,SCGB2A1,ORM1,CALML3,GRPR,FABP4,CHAD,TNNT1,CST5,SORCS1,MAGEA6,KLK7,ASCL1,MUC2,CLIC6,PDZK1,CDC20B,CXCL17,SLC6A14,RPS28,PEG10,DIO1,CARTPT,FOXI1,NBPF4,SPAG6,PYDC1,C7,SDR16C5,CASP14,CNTNAP2,ABCC8,WNK4,MUC15,CIDEC,LCN2,S100A7A,PPAN-P2RY11,SCUBE2,MIA,S100A8,ABCA12,SCGB3A1,PGLYRP2,FOLR1,DSC3,C2orf54,CAPN8,IGSF1,MSMB,CXCL13,S100A9,LRP1B,MAGEA3,SERPINA5,TMPRSS4,LY6D,TRH,SERPINB5,GLYATL2,SLC7A4,HS6ST3,SLC5A1,PON3,MMP1,ERBB4,SLC44A4,SFRP1,NBPF6,PLA2G2A,COL17A1,ALB,ZIC1,S100P,ROBO2,RIMS4,ORM2,CYP4F22,WIF1,NEK10,NCRNA00052,GLRA3,TMC5,ABCC13,IL20,ROPN1,PI16,LPPR3,ELOVL2,INSM1,TSPAN8,CCL19,SLC9A2,SLC7A2,LEP,PI3,LOC728606,FREM2,MS4A15,WFDC2,KNDC1,FAM3B,HGD,ATP6V0A4,WDR72,FGB,CAPN6,CHST8,ATP13A5,NKAIN1,TRPA1,SYT1,NCCRP1,FAM5B,PTPRZ1,ANXA8,CYP4B1,HORMAD1,VTCN1,FOXJ1,CCL21,SYTL5,KLK8,ACTL8,PKP1,CR2,GRB14,SPDYC,KRT15,PLIN1,AGTR1,CHGB,BCAS1,C16orf89,GLDC,TPRG1,BBOX1,TMPRSS6,GPD1,HOTAIR,CYP4X1,TUBA3D,ART3,SYT8,LRG1,GPR98,NAT1,CYP2A7,MS4A1,FSIP1,FAM196A,CIDEA,GSTA1,TRIM29,SLPI,PROL1,NDP,DLK1,CHIT1,ZIC2,FLJ45983,PAX7,ROPN1B,SOSTDC1,HEPACAM2,PPP4R4,PCSK1N,ALOX15B,PNMT,ATRNL1,IGDCC3,CST4,TF,MARCO,PSCA,MMP12,RGS22,ANXA8L2,DSG1,C2CD4A,SLC13A2,LOC100271831,GPR26,AKR7A3,ONECUT2,GABRB3,HOXC11,PRODH,GJB1,KRT23,KRT6C,RBM24,C1orf173,C19orf33,C3orf57,GRIK3,FOXA1,TRPV6,THRSP,UGT2B15,ARHGAP36,DARC,SAA2,SHISA9,KRT4,TMPRSS3,FUT3,VSIG2,IGJ,POF1B,PPP1R14C,SAA1,VIPR2,TPSG1,PRSS21,TDRD1,HBA1,PLIN4,TFPI2,KRT13,AREG,TTYH1,SLC26A3,SLC28A3,PLCH1,MMP13,CCNO,AR,DLX2,DKK1,CXADRP3,SYNPO2L,AFF3,ATP13A4,S100A2,AKR1B10,PI15,SPAG17,NTNG1,FAM83E,SBSN,PCDHA11,ANKRD43,ZNF385B,NPY5R,F7,HSD17B2,DCX,NELL2,MAPK4,FGG,PCDHA12,IYD,SOX2,MYT1,CACNG4,DCDC2,FAM83A,EPYC,ADCY5,TSIX,CHI3L2,PHGR1,PCDH10,WT1,SHISA2,KCNG1,PP14571,EDN3,HPX,RIMS2,COL11A1,CD177,TLX1,PCK1,GLYATL1,CYP4F11,SEZ6L,LOC642587,S100A1,TTC36,MMP7,PART1,NEURL,TBC1D3G,CNTFR,GABBR2,CRYM,TOX3,PPP1R1A,SCRG1,PLA2G2D,LGR6,RET,SLC15A1,ZFP57,SMOC1,HOXC10,TUBA3E,UGT2B28,MKRN3,HRASLS5,SLC6A11,CXCL14,CHGA,PLA2G3,KCNF1,RHCG,CHRDL1,TSPAN1,MKX,BCL11A,FBN3,MS4A8B,SLC6A15,NKX2-2,MAGEA1,FUT6,MAPT,CTNND2,CPLX2,AKR1C2,DOK7,UGT8,PCSK1,LOC389033,CECR2,APOD,OGN,RASAL1,DACH1,ABCC12,HP,TMEM40,MMP10,CPA6,ABCA8,FAT2,SCNN1G,DEFB1,LY6K,C2orf40,SCARA5,FGFBP1,NOVA1,CLGN,FGF10,SCGN,PIK3C2G,CCL18,RBP4,GRM4,TPSD1,ADAMTS15,CD79A,C1orf106,TNFSF12-TNFSF13,FLT3,AQP7,PAH,GPRIN2,PLIN5,NMU,IRX4,ADAMTS19,SHC4,CLSTN2,GATA4,GJB3,B3GNT3,FCRL5,TMEM132C,SLC1A1,EMX1,PCOLCE2,DLGAP1,PSAT1,MAOB,COL9A3,AFP,ADAM6,LOC339535,IRX1,LOC440905,SEMA3E,CD300LG,SERHL2,GNG4,CAPN9,MYO3B,LOC84740,PDZK1IP1,SLC44A5,CXorf61,DNALI1,HAPLN1,PAX2,SLC19A3,CD19,DLX1,SPDEF,C6,SERPINB2,MAGEA4,IL12RB2,UNC5A,CAPN13,CHRM1,COCH,PEG3,CKMT1A,TMEM213,FERMT1,SPINK8,SOX11,NELL1,PADI3,TNNI3,CXCL9,CHI3L1,PCDHA10,PTCHD1,FUT9,CA8,LOC145837,LRRC26,GAL,CWH43,CA12,DNER,RNF183,L1CAM,SLC27A6,STC2,DPYSL5,CDSN,ACTA1,TRIM58,PCDHA6,M

##Complete validation

In [160]:
# ============================================================
# FINAL MASTER DATASET VALIDATION
# ============================================================

print("=" * 70)
print("MASTER DATASET VALIDATION")
print("=" * 70)

print("Rows:")
print(len(master))

print("\nUnique patients:")
print(master["patient_id"].nunique())

print("\nDuplicate patients:")
print(master["patient_id"].duplicated().sum())

print("\nMissing patient IDs:")
print(master["patient_id"].isna().sum())

print("\nMissing survival times:")
print(master["survival_time"].isna().sum())

print("\nMissing event labels:")
print(master["event"].isna().sum())

print("\nNumber of pathology feature files:")
print(master["file_uuid"].notna().sum())

print("\nEvent distribution:")
print(master["event"].value_counts())

print("\nEvent ratio:")
print(round(master["event"].mean(), 4))

MASTER DATASET VALIDATION
Rows:
357

Unique patients:
357

Duplicate patients:
0

Missing patient IDs:
0

Missing survival times:
0

Missing event labels:
0

Number of pathology feature files:
357

Event distribution:
event
0    266
1     91
Name: count, dtype: int64

Event ratio:
0.2549


##Verify every pathology feature exists

In [161]:
# ============================================================
# VERIFY FEATURE FILES EXIST
# ============================================================

missing_files = []

for uuid in master["file_uuid"]:

    path = os.path.join(
        FEATURE_DIR,
        f"{uuid}.npy"
    )

    if not os.path.exists(path):
        missing_files.append(uuid)

print("=" * 70)
print("FEATURE FILE VALIDATION")
print("=" * 70)

print("Missing feature files:")
print(len(missing_files))

if len(missing_files) > 0:
    print(missing_files[:10])
else:
    print("All pathology feature files exist.")

FEATURE FILE VALIDATION
Missing feature files:
0
All pathology feature files exist.


##Check feature dimensions

In [162]:
# ============================================================
# VERIFY FEATURE DIMENSIONS
# ============================================================

sample_size = min(10, len(master))

dimensions = []

for uuid in master["file_uuid"].sample(sample_size, random_state=42):

    feature = np.load(
        os.path.join(FEATURE_DIR, f"{uuid}.npy")
    )

    dimensions.append(feature.shape)

print("=" * 70)
print("FEATURE DIMENSIONS")
print("=" * 70)

for shape in dimensions:
    print(shape)

FEATURE DIMENSIONS
(5733, 512)
(3073, 512)
(964, 512)
(16455, 512)
(5704, 512)
(2393, 512)
(15186, 512)
(6432, 512)
(2471, 512)
(17117, 512)


##Summary report

In [163]:
# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 70)
print("MULTIMODAL DATASET SUMMARY")
print("=" * 70)

print(f"Patients               : {len(master)}")
print(f"Clinical features      : {clinical.shape[1]-1}")
print(f"Genomic features       : {genomic.shape[1]-1}")
print(f"Survival labels        : Yes")
print(f"Histopathology slides  : {master['file_uuid'].notna().sum()}")
print(f"Recurrence events      : {master['event'].sum()}")
print(f"Non-recurrence         : {(master['event']==0).sum()}")
print(f"Event ratio            : {master['event'].mean():.3f}")

print("\nDataset is ready for model development.")

MULTIMODAL DATASET SUMMARY
Patients               : 357
Clinical features      : 20
Genomic features       : 1000
Survival labels        : Yes
Histopathology slides  : 357
Recurrence events      : 91
Non-recurrence         : 266
Event ratio            : 0.255

Dataset is ready for model development.
